In [ ]:
%pip install -q pandas transformers torch tqdm "optimum[onnxruntime]" onnxruntime

Lyrics Emotion Classification — both classifiers
=================================================
Input:  `03_lyrics_trans.csv` (rank, artist, title, region, spotify_uri,
        lyrics_in_en, original_lang)
Output: one CSV per classifier, each with its own emotion score columns plus
        `dominant_emotion`, `dominant_score`, `low_confidence`

Replaces the former `04.1_classification_zeroshot.ipynb` and
`04.2_classification_goemotions.ipynb` (both archived). Everything up to and
including chunking was byte-identical between them, as was the checkpoint
machinery and the output derivation — so it lives here once. Each classifier
keeps its own straight-line pipeline below (config → load → classify_song →
main loop), just like the original two notebooks, so either section reads
top-to-bottom on its own.

| | zero-shot NLI (§ A) | GoEmotions (§ B) |
|---|---|---|
| model | `facebook/bart-large-mnli` | `SamLowe/roberta-base-go_emotions-onnx` |
| labels | 10, hand-picked for songs | 28, fixed (27 emotions + `neutral`) |
| kind | zero-shot NLI entailment | supervised, fine-tuned |
| output | `04.1_emotion_scores_zeroshot.csv` | `04.2_emotion_scores_goemotions.csv` |
| downstream | `05.1` → `06.1` | `05.2` → `06.2` |

Output filenames keep their `04.1` / `04.2` prefixes so nothing downstream had
to change.

Running one classifier, not both
--------------------------------
Set `RUN` below. The two differ a lot in cost (GoEmotions ~29 min; zero-shot NLI
is considerably slower), so "run all cells" should never be the only option.

Shared scoring contract
-----------------------
The reason these two are worth running side by side is that they answer the
same question with different vocabularies. That only holds if everything
*except* model and taxonomy is identical between them:

1. **Independent per-label probabilities in [0, 1].** No softmax across labels,
   so scores do not sum to 1 and a song can score high on several emotions at
   once. GoEmotions does this natively (sigmoid head); zero-shot does it via
   `ZEROSHOT_MULTI_LABEL` (see the note on that setting below — it is a real
   trade-off, not a default to accept unthinkingly).
2. **`unclassified` means "no scoreable lyrics"** (every score 0), never "low
   confidence". Both classifiers therefore drop the same ~107 empty-lyric songs
   and carry the same song set forward.
3. **Confidence is data, not a filter.** `dominant_score` records the winning
   score; `low_confidence` flags it against the shared `MIN_CONFIDENCE`.
   Nothing is dropped on it, so any confidence filter downstream is applied
   identically to both.

Point 3 replaced two different hard-drop rules (`> 0` here, `>= 0.30` there)
that between them accounted for essentially the entire apparent coverage gap
between the classifiers — 108 vs 201 songs dropped, where a common bar drops
the same ~107 from each. Full rationale for all of this in
`docs/classifier_methodology.md`.

`07_compare_classifiers.ipynb` verifies all three before comparing anything.

### Shared config

Only what genuinely has to be identical for the contract to hold. Everything
classifier-specific (label list, model id, hypothesis template, checkpoint
path...) is declared inside that classifier's own section, not here.

In [ ]:
import re
from pathlib import Path

import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"
# DATABRICKS PATH
# PROCESSED = Path("/Volumes/songs_db/default/storage")

# ── Which classifiers to run this session ────────────────────────────────────
# Trim this to skip an expensive run.
RUN = ["zeroshot", "goemotions"]

# ── Shared across BOTH classifiers — keep these two in sync ──────────────────
CHUNK_WORDS = 350       # words per chunk; both models take 512 tokens
MIN_CONFIDENCE = 0.30   # confidence bar for the low_confidence FLAG (drops nothing)
CHECKPOINT_EVERY = 10   # songs between checkpoint flushes

analysis_df = pd.read_csv(PROCESSED / "03_lyrics_trans.csv")
if "lyrics_in_en" not in analysis_df.columns:
    raise KeyError("Expected 'lyrics_in_en' column in 03_lyrics_trans.csv")

print(f"Loaded {len(analysis_df)} songs")
display(analysis_df.head(3))

### 1. Cleaning — shared, model-agnostic

In [ ]:
def clean_lyrics(text: str, artist: str = "") -> str:
    """
    Remove structural noise but preserve punctuation and casing.
    Punctuation (! ? ...) and capitalisation carry emotional signal for both
    classifiers — don't strip them.

    artist: the artist string from the dataframe row. When provided, the first
    line is dropped only if its tokens are a subset of the known artist names —
    much more precise than a regex heuristic. Falls back to the regex heuristic
    when artist is unavailable.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    def is_artist_credit(line: str) -> bool:
        """
        True if every name token in `line` exists in the artist pool.
        Both strings are split on commas, ampersands, and feat/ft.

            artist = "Jason, Bonnie"          line = "Jason"          → True
            artist = "ARIA VEGA, Ryan Castro" line = "Ryan Castro"    → True
            artist = "Jason, Bonnie"          line = "Baby come back" → False
        """
        splitter = r"[,&]|\bfeat\.?\b|\bft\.?\b"
        artist_tokens = {
            t.strip().lower()
            for t in re.split(splitter, artist, flags=re.IGNORECASE)
            if t.strip()
        }
        line_tokens = {
            t.strip().lower()
            for t in re.split(splitter, line, flags=re.IGNORECASE)
            if t.strip()
        }
        return bool(line_tokens) and line_tokens.issubset(artist_tokens)

    # Remove section headers: [Verse 1], [Chorus], [Bridge] etc.
    text = re.sub(r"\[[^\]]*\]", "", text)

    # Remove repetition annotations: (x3), (×2), (2x)
    text = re.sub(r"\([\d×xX]+\)", "", text)

    # Remove leading artist/feature credits that sometimes appear at the top of
    # translated lyrics (e.g. "ARIA VEGA, Ryan Castro\n").
    lines = text.strip().splitlines()
    if lines:
        first = lines[0].strip()
        if artist and is_artist_credit(first):
            lines = lines[1:]
        elif not artist and re.match(r"^[A-Za-z\s,&]+$", first) and len(first) < 80:
            lines = lines[1:]
    text = "\n".join(lines)

    # Collapse excessive blank lines but keep single line breaks
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


def dedupe_lines(text: str) -> str:
    """
    Remove duplicate lines (repeated choruses inflate scores).
    Keeps first occurrence; preserves order.
    """
    seen = set()
    result = []
    for line in text.splitlines():
        key = line.strip().lower()
        if key and key not in seen:
            seen.add(key)
            result.append(line.strip())
    return " ".join(result)


def prepare_lyrics(text: str, artist: str = "") -> str:
    return dedupe_lines(clean_lyrics(text, artist=artist))

### 2. Chunking — shared

In [ ]:
def chunk_text(text: str, chunk_words: int = CHUNK_WORDS) -> list[str]:
    """Split text into word-count chunks with a small overlap."""
    words = text.split()
    if not words:
        return []
    overlap = 30
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_words
        chunks.append(" ".join(words[start:end]))
        start = end - overlap  # small overlap so context isn't lost at boundaries
        if start >= len(words):
            break
    return chunks

### 3. Checkpointing + contract derivation — shared

Generic over `emotion_cols` / `checkpoint_path` so both sections below call the
same three functions, but each section still owns its own main loop — nothing
here decides *when* or *what* to classify.

In [ ]:
def load_checkpoint(checkpoint_path: Path, emotion_cols: list[str]) -> pd.DataFrame | None:
    """
    Load an existing checkpoint CSV if it exists, else None.

    NOTE: a COMPLETE checkpoint makes classification a no-op — only rows with
    NaN scores get classified, so changing a SCORING setting and re-running
    silently reuses the old scores. This is exactly how this notebook came to
    disagree with its own data once already (04.1's checkpoint kept
    multi_label=True-shaped scores after the code was edited to
    multi_label=False — see docs/classifier_methodology.md). Delete the
    checkpoint when you change the taxonomy, the model, the chunking, or
    ZEROSHOT_MULTI_LABEL. Changing MIN_CONFIDENCE is safe — it's applied below,
    which always re-runs.
    """
    if not checkpoint_path.exists():
        return None
    df = pd.read_csv(checkpoint_path)
    done = df[emotion_cols].notna().all(axis=1).sum()
    print(f"Checkpoint found: {done} / {len(df)} songs already classified.")
    if done == len(df):
        print("  ⚠ Checkpoint is COMPLETE — no song will be re-scored this run.")
        print("    Delete it first if you changed how scores are PRODUCED.")
    return df


def save_checkpoint(df: pd.DataFrame, checkpoint_path: Path) -> None:
    """Write current state (including any NaN emotion cols) to disk."""
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(checkpoint_path, index=False)


def derive_contract_columns(df: pd.DataFrame, emotion_cols: list[str]) -> pd.DataFrame:
    """
    Contract points 2 and 3 — called identically by both sections below.

    Scores are independent per-label probabilities, so the dominant emotion is
    just the argmax; there is no competition to resolve. 'unclassified' is
    reserved for songs with NO scoreable lyrics (every score 0). Low confidence
    is recorded and flagged, never dropped — dropping on a per-classifier
    threshold is what previously left the two classifiers holding different
    song sets and made a config difference look like a model difference.
    """
    df = df.copy()
    df[emotion_cols] = df[emotion_cols].astype(float)

    max_score = df[emotion_cols].max(axis=1)
    has_signal = max_score > 0

    df["dominant_emotion"] = (
        df[emotion_cols].idxmax(axis=1).str.replace("emotion_", "", regex=False)
          .where(has_signal, other="unclassified")
    )
    df["dominant_score"] = max_score.where(has_signal)
    df["low_confidence"] = has_signal & (max_score < MIN_CONFIDENCE)

    print(f"unclassified (no scoreable lyrics): {(~has_signal).sum()} / {len(df)}")
    print(f"low_confidence (scored, but < {MIN_CONFIDENCE}): "
          f"{df['low_confidence'].sum()} / {len(df)}")
    return df

---
## § A. Zero-Shot NLI — `facebook/bart-large-mnli`

Custom, song-specific emotion vocabulary.

In [ ]:
# ── Config: zero-shot ─────────────────────────────────────────────────────────
ZEROSHOT_EMOTIONS = [
    "love", "longing", "joy", "heartbreak", "grief",
    "despair", "hope", "lonely",
    "sensual", "anger",
]
zeroshot_emotion_cols = [f"emotion_{e}" for e in ZEROSHOT_EMOTIONS]

# The sentence each candidate label is slotted into before the NLI model judges
# entailment against the lyrics. Phrasing it as "the dominant emotion" keeps the
# model judging song-level affect rather than whether the word merely appears.
HYPOTHESIS_TEMPLATE = "The dominant emotion in this song is {}."

# ── ZEROSHOT_MULTI_LABEL — a decision worth re-reading before you run ────────
# `local/progress.md` records a 2026-07-12 decision to set this False; it is
# now True. Both positions are legitimate — see docs/classifier_methodology.md
# § "Learning point 4" for the full argument. Short version:
#
#   False (softmax across labels, the old setting): fixes score INFLATION —
#   with independent scoring, `longing`/`sensual` average 0.74-0.77 corpus-wide
#   and win 575/1105 songs between them, so "dominant emotion" partly reports
#   which label bart-mnli over-estimates globally, not what's distinctive
#   about a song.
#
#   True (independent per-label, current): avoids two problems with softmax —
#   (1) scores become an artifact of the label list (add an 11th emotion, all
#   ten existing scores move), and (2) it's the only setting under which this
#   classifier and GoEmotions emit the same KIND of quantity (contract point 1),
#   which is required for 07_compare_classifiers to mean anything.
#
#   Inflation under True is handled downstream instead: 06.1 § 4 and 07 § 5
#   z-score each label against its own regional baseline, which removes exactly
#   that global per-label offset — the same idea the 2026-07-12 differential
#   heatmap was already reaching for, just applied at analysis time rather than
#   baked into the scoring step.
#
# If you set this to False, everything still runs: 07 detects the softmax and
# restricts itself to rank-based comparisons rather than comparing incomparable
# magnitudes.
ZEROSHOT_MULTI_LABEL = True

zeroshot_checkpoint_path = PROCESSED / "04.1_emotion_scores_zeroshot_checkpoint.csv"
zeroshot_output_path = PROCESSED / "04.1_emotion_scores_zeroshot.csv"
# DATABRICKS PATHS
# zeroshot_checkpoint_path = Path("/Volumes/songs_db/default/storage/04.1_emotion_scores_zeroshot_checkpoint.csv")
# zeroshot_output_path = Path("/Volumes/songs_db/default/storage/04.1_emotion_scores_zeroshot.csv")

In [ ]:
def load_zeroshot_classifier():
    """
    facebook/bart-large-mnli — best general zero-shot NLI classifier.
    Switch to cross-encoder/nli-deberta-v3-large for higher accuracy at the
    cost of ~2× slower inference.
    """
    from transformers import pipeline

    print("Loading zero-shot classifier (bart-large-mnli)...")
    try:
        return pipeline("zero-shot-classification",
                        model="facebook/bart-large-mnli", device=0)
    except Exception as e:
        print(f"GPU load failed ({e}); falling back to CPU...")
        return pipeline("zero-shot-classification",
                        model="facebook/bart-large-mnli", device=-1)


def classify_song_zeroshot(lyrics: str, classifier) -> dict:
    """
    Classify a single song's lyrics.

    With ZEROSHOT_MULTI_LABEL=True, each label is scored independently — a
    softmax over that label's own (contradiction, entailment) pair — giving a
    probability in [0, 1] per emotion that does NOT compete with the others.
    Scores therefore do not sum to 1, and a song can legitimately score high on
    `longing` and `sensual` at once. See the ZEROSHOT_MULTI_LABEL note above for
    why this is the setting, not a fallback.

    Chunk scores are averaged to produce a song-level score per emotion.

    Returns dict {emotion: score}.
    """
    BATCH_SIZE = 16

    if not isinstance(lyrics, str) or not lyrics.strip():
        return {e: 0.0 for e in ZEROSHOT_EMOTIONS}

    chunks = chunk_text(lyrics)
    raw_results = []

    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i : i + BATCH_SIZE]
        raw_results.extend(
            classifier(
                batch,
                candidate_labels=ZEROSHOT_EMOTIONS,
                multi_label=ZEROSHOT_MULTI_LABEL,
                hypothesis_template=HYPOTHESIS_TEMPLATE,
            )
        )

    # The pipeline returns labels sorted by score, so zip back into a dict
    # rather than assuming ZEROSHOT_EMOTIONS order.
    all_scores = [dict(zip(r["labels"], r["scores"])) for r in raw_results]

    return {
        e: round(sum(s[e] for s in all_scores) / len(all_scores), 4)
        for e in ZEROSHOT_EMOTIONS
    }

### Main pipeline — zero-shot

In [ ]:
if "zeroshot" not in RUN:
    print("Skipped (not in RUN).")
else:
    df_zs = load_checkpoint(zeroshot_checkpoint_path, zeroshot_emotion_cols)
    if df_zs is None:
        df_zs = analysis_df.copy()
        for col in zeroshot_emotion_cols:
            if col not in df_zs.columns:
                df_zs[col] = pd.NA

    if "lyrics_prepared" not in df_zs.columns:
        df_zs["lyrics_prepared"] = df_zs.apply(
            lambda r: prepare_lyrics(r.get("lyrics_in_en", ""), artist=r.get("artist", "")),
            axis=1,
        )

    todo = df_zs[df_zs[zeroshot_emotion_cols].isna().any(axis=1)].index.tolist()
    print(f"Songs pending classification: {len(todo)} / {len(df_zs)}")

    if todo:
        zeroshot_classifier = load_zeroshot_classifier()
        n_saved = 0
        for i, idx in enumerate(tqdm(todo, desc="zeroshot", unit="song")):
            scores = classify_song_zeroshot(df_zs.at[idx, "lyrics_prepared"], zeroshot_classifier)
            for emotion, score in scores.items():
                df_zs.at[idx, f"emotion_{emotion}"] = score

            n_saved += 1
            if n_saved % CHECKPOINT_EVERY == 0:
                save_checkpoint(df_zs, zeroshot_checkpoint_path)
                tqdm.write(f"  ✓ Checkpoint saved ({n_saved} songs this run)")

        save_checkpoint(df_zs, zeroshot_checkpoint_path)
        print(f"Done. Checkpoint saved to {zeroshot_checkpoint_path}")
    else:
        print("No unclassified rows. Using existing scores.")

    df_zs = derive_contract_columns(df_zs, zeroshot_emotion_cols)
    df_zs = df_zs.drop(columns=["lyrics_prepared"])
    df_zs.to_csv(zeroshot_output_path, index=False)
    print(f"Saved to {zeroshot_output_path}")
    display(df_zs.head(3))

---
## § B. GoEmotions — `SamLowe/roberta-base-go_emotions-onnx`

Supervised, fixed 28-label taxonomy, served through ONNX Runtime.

In [ ]:
# ── Config: GoEmotions ────────────────────────────────────────────────────────
# GoEmotions' fixed taxonomy (27 emotions + neutral). Unlike ZEROSHOT_EMOTIONS,
# this is not customizable without fine-tuning — it's whatever the model was
# trained on. `neutral` has no zero-shot counterpart; handled downstream in 05.2.
GOEMOTIONS_EMOTIONS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral",
]
goemotions_emotion_cols = [f"emotion_{e}" for e in GOEMOTIONS_EMOTIONS]

GOEMOTIONS_MODEL_ID = "SamLowe/roberta-base-go_emotions-onnx"

goemotions_checkpoint_path = PROCESSED / "04.2_emotion_scores_goemotions_checkpoint.csv"
goemotions_output_path = PROCESSED / "04.2_emotion_scores_goemotions.csv"
# DATABRICKS PATHS
# goemotions_checkpoint_path = Path("/Volumes/songs_db/default/storage/04.2_emotion_scores_goemotions_checkpoint.csv")
# goemotions_output_path = Path("/Volumes/songs_db/default/storage/04.2_emotion_scores_goemotions.csv")

In [ ]:
def load_goemotions_classifier():
    """
    RoBERTa fine-tuned on GoEmotions, exported to ONNX and served via optimum's
    ONNX Runtime backend. One supervised forward pass per input (no NLI
    hypothesis pairs), so substantially faster than the zero-shot classifier.

    top_k=None returns all 28 label scores per input. function_to_apply='sigmoid'
    matches how the model was trained: multi-label, independent per-label
    probabilities — contract point 1, natively.
    """
    from optimum.onnxruntime import ORTModelForSequenceClassification
    from transformers import AutoTokenizer, pipeline

    print(f"Loading GoEmotions ONNX classifier ({GOEMOTIONS_MODEL_ID})...")
    model = ORTModelForSequenceClassification.from_pretrained(GOEMOTIONS_MODEL_ID)
    tokenizer = AutoTokenizer.from_pretrained(GOEMOTIONS_MODEL_ID)
    return pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        top_k=None,
        function_to_apply="sigmoid",
    )


def classify_song_goemotions(lyrics: str, classifier) -> dict:
    """
    Classify a single song's lyrics.

    Each chunk gets independent sigmoid scores for all 28 labels — a song can
    score high on several emotions at once, and scores do not sum to 1 (same
    behaviour as zero-shot under ZEROSHOT_MULTI_LABEL=True; the two differ in
    model and label set, not in scoring mechanics). Chunk scores are averaged to
    a song-level score.

    Note these sigmoids sit systematically lower than bart-mnli's entailment
    probabilities (median top score ~0.50 vs ~0.97). Same kind of quantity under
    the contract, but not the same scale — cross-classifier reads should go
    through the z-scored views in 06.1/06.2 § 4 or 07 § 5.

    Chunks are classified one at a time, NOT batched. This ONNX export has a
    fixed-batch assumption in its position-embedding broadcast — passing a list
    of >1 texts triggers "INVALID_ARGUMENT ... Expand node ... invalid expand
    shape" from onnxruntime. Single-example calls avoid the broadcast entirely.

    Whether a single-string call returns a flat list of 28 dicts or a nested
    [[...]] list varies by transformers/optimum version, so unwrap defensively.

    Returns dict {emotion: score}.
    """
    if not isinstance(lyrics, str) or not lyrics.strip():
        return {e: 0.0 for e in GOEMOTIONS_EMOTIONS}

    all_scores = []
    for chunk in chunk_text(lyrics):
        result = classifier(chunk, truncation=True)
        if isinstance(result, list) and result and isinstance(result[0], list):
            result = result[0]  # unwrap [[{...}, ...]] -> [{...}, ...]
        all_scores.append({d["label"]: d["score"] for d in result})

    return {
        e: round(sum(s.get(e, 0.0) for s in all_scores) / len(all_scores), 4)
        for e in GOEMOTIONS_EMOTIONS
    }

### Main pipeline — GoEmotions

In [ ]:
if "goemotions" not in RUN:
    print("Skipped (not in RUN).")
else:
    df_ge = load_checkpoint(goemotions_checkpoint_path, goemotions_emotion_cols)
    if df_ge is None:
        df_ge = analysis_df.copy()
        for col in goemotions_emotion_cols:
            if col not in df_ge.columns:
                df_ge[col] = pd.NA

    if "lyrics_prepared" not in df_ge.columns:
        df_ge["lyrics_prepared"] = df_ge.apply(
            lambda r: prepare_lyrics(r.get("lyrics_in_en", ""), artist=r.get("artist", "")),
            axis=1,
        )

    todo = df_ge[df_ge[goemotions_emotion_cols].isna().any(axis=1)].index.tolist()
    print(f"Songs pending classification: {len(todo)} / {len(df_ge)}")

    if todo:
        goemotions_classifier = load_goemotions_classifier()
        n_saved = 0
        for i, idx in enumerate(tqdm(todo, desc="goemotions", unit="song")):
            scores = classify_song_goemotions(df_ge.at[idx, "lyrics_prepared"], goemotions_classifier)
            for emotion, score in scores.items():
                df_ge.at[idx, f"emotion_{emotion}"] = score

            n_saved += 1
            if n_saved % CHECKPOINT_EVERY == 0:
                save_checkpoint(df_ge, goemotions_checkpoint_path)
                tqdm.write(f"  ✓ Checkpoint saved ({n_saved} songs this run)")

        save_checkpoint(df_ge, goemotions_checkpoint_path)
        print(f"Done. Checkpoint saved to {goemotions_checkpoint_path}")
    else:
        print("No unclassified rows. Using existing scores.")

    df_ge = derive_contract_columns(df_ge, goemotions_emotion_cols)
    df_ge = df_ge.drop(columns=["lyrics_prepared"])
    df_ge.to_csv(goemotions_output_path, index=False)
    print(f"Saved to {goemotions_output_path}")
    display(df_ge.head(3))

# ~29m runtime for a full GoEmotions pass.

---
### Regional summaries

Fork-local averages per classifier that ran. See `06.1`/`06.2` § 4 for the
z-scored views that survive a cross-classifier read.

In [ ]:
titles_df = pd.read_csv(PROCESSED / "00_titles.csv")[["spotify_uri", "region"]]

if "zeroshot" in RUN:
    merged = titles_df.merge(df_zs[["spotify_uri"] + zeroshot_emotion_cols],
                              on="spotify_uri", how="inner")
    zeroshot_regional_summary = merged.groupby("region")[zeroshot_emotion_cols].mean().round(3)
    zeroshot_regional_summary.columns = [c.replace("emotion_", "") for c in zeroshot_regional_summary.columns]
    display(zeroshot_regional_summary)
    out_path = PROCESSED / "04.1_regional_summary_zeroshot.csv"
    zeroshot_regional_summary.to_csv(out_path, index=False)
    print(f"Saved to {out_path}")

if "goemotions" in RUN:
    merged = titles_df.merge(df_ge[["spotify_uri"] + goemotions_emotion_cols],
                              on="spotify_uri", how="inner")
    goemotions_regional_summary = merged.groupby("region")[goemotions_emotion_cols].mean().round(3)
    goemotions_regional_summary.columns = [c.replace("emotion_", "") for c in goemotions_regional_summary.columns]
    display(goemotions_regional_summary)
    out_path = PROCESSED / "04.2_regional_summary_goemotions.csv"
    goemotions_regional_summary.to_csv(out_path, index=False)
    print(f"Saved to {out_path}")